In [14]:
%load_ext autoreload
%autoreload 2

# function to calculate lick runs with more options

from pathlib import Path
import numpy as np
import trompy as tp
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:

DATAFOLDER = Path("D:/TestData/distraction/medfiles/")

medfile = DATAFOLDER / "!2017-06-20_10h01m.Subject thph1.3"

medfile = Path("C:/Users/jmc010/Github/lickcalc-paper/data/2021-06-17_10h00m_Subject ARP306.txt")

# medfile = Path("C:/Github/lickcalc_webapp/data/!2017-07-28_09h29m.Subject pcf1.03/")


onset = tp.medfilereader(medfile, varsToExtract=["e"])[1:]
offset = tp.medfilereader(medfile, varsToExtract=["f"])[1:]

lickdata = tp.lickcalc(onset, offset=offset)
lickdata["licklength_mode"]
lickdata["intraburst_mode"]
lickdata["intercontact_mode"]


Hey there [33.565, 33.677, 34.007, 34.152, 34.297, 34.534, 34.622, 34.716, 34.812, 34.844, 34.914, 35.02, 35.235, 35.344, 35.449, 35.561, 35.68, 35.8, 35.914, 36.036, 36.261, 36.376, 36.493, 36.609, 36.729, 36.857, 36.975, 37.1, 37.33, 37.447, 37.567, 37.689, 37.812, 37.932, 38.056, 38.179, 38.302, 38.426, 38.554, 38.674, 38.798, 39.039, 39.156, 39.28, 39.4, 39.521, 39.641, 39.763, 39.882, 39.991, 40.216, 40.332, 40.451, 40.571, 40.695, 40.927, 41.051, 41.175, 41.294, 41.413, 41.542, 41.774, 41.894, 42.012, 42.247, 42.358, 42.481, 42.714, 42.841, 42.958, 43.084, 43.34, 43.462, 43.585, 43.818, 43.939, 44.056, 44.288, 44.407, 44.534, 44.781, 44.897, 45.015, 45.253, 45.375, 45.732, 45.84, 47.148, 47.251, 47.378, 47.482, 47.6, 47.716, 47.841, 48.059, 48.166, 48.285, 48.408, 48.64, 48.763, 48.895, 49.022, 49.151, 49.274, 49.398, 49.52, 49.649, 49.777, 49.904, 50.024, 50.159, 50.283, 50.411, 50.534, 50.652, 50.913, 51.038, 51.284, 51.4, 51.533, 51.659, 63.523, 63.617, 63.718, 63.927, 64.037,

0.064

In [18]:
lickdata = tp.lickcalc(onset)
lickdata["intercontact_time"]

Hey there []


In [9]:
L = tp.Lickcalc(licks=onset, offset=offset)
L.intercontact_mode

0.053

In [6]:
tp.Lickcalc??

Init signature: tp.Lickcalc(**kwargs)
Source:        
class Lickcalc:
    """
    Analyzes licking behavior data to compute bursts, runs, and related statistics.

    This class processes raw lick onset and offset times to extract comprehensive
    behavioral metrics including burst structure, lick runs, inter-lick intervals,
    and burst probability distributions.

    Parameters
    ----------
    licks : array_like
        Lick onset times (in seconds or arbitrary time units).
    offset : array_like, optional
        Lick offset times. Required for lick length analysis.
    longlick_threshold : float, default 0.3
        Duration threshold (seconds) above which licks are classified as "long".
    burst_threshold : float, default 0.5
        Inter-lick interval threshold (seconds) for burst boundaries.
    min_burst_length : int, default 1
        Minimum number of licks to define a burst. Shorter bursts are filtered.
    run_threshold : float, default 10
        Time threshold (se

In [7]:
intercontact_time = np.array(offset) - np.array(onset)

def get_mode(data, binsize=0.001, smooth_window=20):
    hist = np.histogram(data, bins=np.arange(0, np.max(data) + binsize, binsize))
    hist_smoothed = pd.Series(hist[0]).rolling(smooth_window, center=True).mean()
    return hist_smoothed.idxmax() * binsize

get_mode(intercontact_time, binsize=0.001, smooth_window=20)

0.053

In [53]:
L = tp.Lickcalc(licks=onset, burst_threshold=1)
first_burst_ilis = L.get_first_n_ilis_in_bursts(n_ilis=5,
                                                #pre_ili=1,
                                                #min_ili=0.1
                                                burst_index=50
                                                )

In [52]:
first_burst_ilis

ili_index
0    0.232333
1    0.289750
2    0.126000
3    0.103500
4    0.107500
Name: ili, dtype: float64

In [1]:
offset - onset

NameError: name 'offset' is not defined

In [31]:
burst_df = L.get_ilis_in_bursts()

In [38]:


(burst_df
 .query("pre_ili > 4")
.query("ili > 0.06")
.query("burst_index < 10")
.groupby("ili_index")
.mean()
.iloc[:5]
)
                

        # return (burst_df
        #         .query("pre_ili > @pre_ili")
        #         .query("ili > @min_ili")
        #         .groupby("ili_index")
        #         .mean()
        #         .iloc[:n_ilis]
        #         .ili
        #         )

,burst_index,ili,pre_ili,post_ili
ili_index,,,,
0,4.333333,0.096333,16.623000,17.246333
1,4.666667,0.167667,24.151333,5.426333
2,4.666667,0.133000,24.151333,5.426333
3,4.666667,0.106000,24.151333,5.426333
4,4.666667,0.109000,24.151333,5.426333


In [ ]:
    def get_first_n_ilis_in_bursts(self, n_ilis=5, pre_ili=4, min_ili=0.06):

        burst_df = self.get_ilis_in_bursts()

        return (burst_df
                .query("pre_ili > @pre_ili")
                .query("ili > @min_ili")
                .groupby("ili_index")
                .mean()
                .iloc[:n_ilis]
                .ili
                )

In [26]:
first_burst_ilis

ili_index
0     0.196960
1     0.185318
2     0.136182
3     0.152857
4     0.145136
5     0.116045
6     0.127500
7     0.129143
8     0.118900
9     0.157048
10    0.113400
11    0.118750
12    0.129200
13    0.135450
14    0.160421
15    0.138722
16    0.149368
17    0.139053
18    0.126105
19    0.130722
20    0.127000
21    0.142368
22    0.148105
23    0.131684
24    0.157500
25    0.125667
26    0.135188
27    0.178437
28    0.115813
29    0.145125
30    0.127125
31    0.128812
32    0.160438
33    0.181125
34    0.124875
35    0.136750
36    0.218187
37    0.134062
38    0.140875
39    0.182125
40    0.175312
41    0.141200
42    0.132267
43    0.117067
44    0.139067
45    0.148733
46    0.128333
47    0.136286
48    0.123357
49    0.125857
Name: ili, dtype: float64

In [8]:
tp.Lickcalc??

Init signature: tp.Lickcalc(**kwargs)
Source:        
class Lickcalc:
    """
    Analyzes licking behavior data to compute bursts, runs, and related statistics.

    This class processes raw lick onset and offset times to extract comprehensive
    behavioral metrics including burst structure, lick runs, inter-lick intervals,
    and burst probability distributions.

    Parameters
    ----------
    licks : array_like
        Lick onset times (in seconds or arbitrary time units).
    offset : array_like, optional
        Lick offset times. Required for lick length analysis.
    longlick_threshold : float, default 0.3
        Duration threshold (seconds) above which licks are classified as "long".
    burst_threshold : float, default 0.5
        Inter-lick interval threshold (seconds) for burst boundaries.
    min_burst_length : int, default 1
        Minimum number of licks to define a burst. Shorter bursts are filtered.
    run_threshold : float, default 10
        Time threshold (se

In [39]:
L = tp.Lickcalc(licks=onset, burst_threshold=1)
L.get_ilis_in_bursts().burst_index.max()
L.get_first_n_ilis_in_bursts(n_ilis=5)

ili_index
0    0.194292
1    0.183952
2    0.135714
3    0.145800
4    0.145238
Name: ili, dtype: float64

In [4]:
csvfile = Path("C:/Users/jmc010/Github/lickcalc_webapp/data/test_with_2_bursts.csv")

df = pd.read_csv(csvfile)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\jmc010\\Github\\lickcalc_webapp\\data\\test_with_2_bursts.csv'

In [3]:
lickdata = tp.lickcalc(onset, offset, time_divisions=3)

In [6]:
lickdata["time_divisions"][1]

{'total_licks': 508,
 'intraburst_freq': np.float64(7.138463299143096),
 'n_bursts': 9,
 'mean_licks_per_burst': np.float64(56.44444444444444),
 'weibull_alpha': np.float64(0.04534075992725116),
 'weibull_beta': np.float64(0.5606769398507732),
 'weibull_rsq': np.float64(0.9061795111520979),
 'n_long_licks': 0,
 'max_lick_duration': np.float64(0.2579999999998108),
 'division_type': 'time',
 'division_number': 2,
 'start_time': np.float64(1094.011),
 'end_time': np.float64(2154.492),
 'duration': np.float64(1060.4810000000002)}

In [4]:
allvars

[[3228.0],
 [-1.0,
  28.30999999999999,
  28.40999999999999,
  28.51799999999999,
  28.65199999999999,
  28.803999999999988,
  29.03799999999999,
  29.175999999999988,
  29.300999999999988,
  29.431999999999988,
  29.564999999999987,
  29.69199999999999,
  29.82299999999999,
  29.95599999999999,
  30.090999999999987,
  30.244999999999987,
  30.366999999999987,
  30.50199999999999,
  30.63599999999999,
  30.770999999999987,
  30.90399999999999,
  31.037999999999986,
  31.169999999999987,
  31.30499999999999,
  31.456999999999987,
  31.585999999999988,
  31.72299999999999,
  31.860999999999986,
  31.999999999999986,
  32.14499999999999,
  32.283999999999985,
  32.42599999999999,
  32.57099999999999,
  32.72799999999999,
  32.86399999999999,
  33.00799999999999,
  33.14499999999999,
  33.283999999999985,
  33.42499999999999,
  33.563999999999986,
  33.70699999999999,
  33.850999999999985,
  34.01099999999999,
  34.14699999999999,
  34.29199999999999,
  34.44299999999999,
  34.583999999999

In [ ]:
len(licks)

In [ ]:
tp.lickCalc(licks)["rNum"]

In [ ]:
def get_lick_runs(licks, t_threshold=10, min_licks=3, verbose=False):
    
    licks = np.insert(np.array(licks), 0, 0)
    
    run_start = [(idx, lick) for idx, lick in enumerate(licks) if lick - licks[idx-1] > 10]
    run_end = [(idx, lick) for idx, lick in enumerate(licks[:-1]) if licks[idx+1] - lick > 10][1:]
    run_end.append((len(licks)-1, licks[-1]))

    runs_t, runs_length = [], []
    removed = 0
    for start, end in zip(run_start, run_end):
        if verbose: print(start, end)
            
        if end[0] - start[0] > min_licks-1:
            runs_t.append(start[1])
            runs_length.append(end[0] - start[0] + 1)
        else:
            removed += 1
            
    print(f"Removed {removed} runs because too short.")

    return runs_t, runs_length
             
r, l = get_lick_runs(licks, verbose=True)

In [ ]:

        
        counter = counter + 1